In [1]:
!pip install -q huggingface_hub pyarrow

import os
import json
import numpy as np
import pandas as pd

from google.colab import userdata
from huggingface_hub import list_repo_files, hf_hub_download

HF_TOKEN = userdata.get("HF_TOKEN")

if HF_TOKEN is None:
    raise ValueError("HF_TOKEN is not set. Add it in Colab Secrets first.")

repo_id = "FlyRank/internship-warehouse"
target_table = "fact_content_daily_performance"
target_month = "month=2026-03"

files = list_repo_files(
    repo_id,
    repo_type="dataset",
    token=HF_TOKEN
)

march_files = [
    f for f in files
    if target_table in f
    and target_month in f
    and f.endswith(".parquet")
]

print("March files found:", len(march_files))
march_files[:10]

March files found: 1


['fact_content_daily_performance/month=2026-03/data_0.parquet']

In [2]:
local_files = []

for file_path in march_files:
    local_path = hf_hub_download(
        repo_id=repo_id,
        filename=file_path,
        repo_type="dataset",
        token=HF_TOKEN
    )
    local_files.append(local_path)

df = pd.concat(
    [pd.read_parquet(path) for path in local_files],
    ignore_index=True
)

print("Rows loaded:", len(df))
print("Columns:", len(df.columns))

df.head()

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Rows loaded: 9841378
Columns: 30


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,None,20,0,67,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,None,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,None,125,1,616,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,None,7,0,28,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,None,11,0,25,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
# Use only rows where GSC data is available
work_df = df[df["gsc_data_available"].eq(True)].copy()

# Create CTR safely
work_df["gsc_ctr"] = np.where(
    work_df["gsc_impressions"] > 0,
    work_df["gsc_clicks"] / work_df["gsc_impressions"],
    0
)

# Keep only rows with search impressions
work_df = work_df[work_df["gsc_impressions"] > 0].copy()

print("Rows after GSC available and impressions > 0:", len(work_df))

work_df[
    [
        "content_hash_id",
        "report_date",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "gsc_ctr",
        "ga4_sessions",
        "ga4_engaged_sessions",
    ]
].head()

Rows after GSC available and impressions > 0: 3611061


,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,gsc_ctr,ga4_sessions,ga4_engaged_sessions
0,content_b7e512995f79d5a6,2026-03-01,20,0,3.350000,0.000,NaN,NaN
1,content_05597932fe4da067,2026-03-01,1,0,0.000000,0.000,NaN,NaN
2,content_7a105f548d9c6916,2026-03-01,125,1,4.928000,0.008,NaN,NaN
3,content_905aa32a0230694e,2026-03-01,7,0,4.000000,0.000,NaN,NaN
4,content_a3ea9792f793ec72,2026-03-01,11,0,2.272727,0.000,NaN,NaN


## Lane confirmation

My locked lane is **Lane 2: Refresh / Content Opportunity Scoring**.

For this baseline, I am building a simple rule that finds content pages with search visibility but weak click performance. The rule supports a content review action: check the title, meta description, search intent, and page content for possible refresh or improvement.

This is only a baseline rule. It uses observable search and engagement signals available at the decision moment. It does not use future-window labels or label-derived inputs.

In [5]:
# Signal Check 1: Volume / visibility signal
# Bucket pages by GSC impressions and inspect clicks and CTR.

volume_check_df = work_df.copy()

volume_check_df["impression_bucket"] = pd.qcut(
    volume_check_df["gsc_impressions"],
    q=4,
    labels=["low", "medium", "high", "very_high"],
    duplicates="drop"
)

volume_bucket_table = (
    volume_check_df
    .groupby("impression_bucket", observed=True)
    .agg(
        n=("content_hash_id", "count"),
        median_impressions=("gsc_impressions", "median"),
        median_clicks=("gsc_clicks", "median"),
        mean_ctr=("gsc_ctr", "mean"),
    )
    .reset_index()
)

volume_bucket_table

,impression_bucket,n,median_impressions,median_clicks,mean_ctr
0,low,973112,2.0,0.0,0.004055
1,medium,868861,9.0,0.0,0.002362
2,high,873000,31.0,0.0,0.002730
3,very_high,896088,150.0,0.0,0.003062


### Signal 1 verdict: CONFIRMED

The volume signal is confirmed enough for a baseline because impression buckets show real differences in search visibility. For my lane, this matters because a page with more impressions has more visible opportunity. If a page is visible but under-clicked, it may deserve review before a page with almost no search exposure.

In [7]:
# Signal Check 2: CTR vs average position
# Bucket average position and inspect CTR.

position_check_df = work_df.copy()

position_check_df["position_bucket"] = pd.cut(
    position_check_df["gsc_avg_position"],
    bins=[0, 3, 10, 20, 1000],
    labels=["top_3", "page_1_4_to_10", "page_2_11_to_20", "deep_20_plus"],
    include_lowest=True
)

position_bucket_table = (
    position_check_df
    .groupby("position_bucket", observed=True)
    .agg(
        n=("content_hash_id", "count"),
        median_position=("gsc_avg_position", "median"),
        median_impressions=("gsc_impressions", "median"),
        median_ctr=("gsc_ctr", "median"),
        mean_ctr=("gsc_ctr", "mean"),
    )
    .reset_index()
)

position_bucket_table

,position_bucket,n,median_position,median_impressions,median_ctr,mean_ctr
0,top_3,727362,1.500000,12.0,0.0,0.004756
1,page_1_4_to_10,1456122,5.868601,23.0,0.0,0.003473
2,page_2_11_to_20,519223,14.000000,20.0,0.0,0.002770
3,deep_20_plus,908354,37.086957,9.0,0.0,0.001289


### Signal 2 verdict: CONFIRMED

The CTR-vs-position signal is confirmed enough for a baseline because CTR should be interpreted together with ranking position. A low CTR is more concerning when a page already has search visibility and a reasonable average position. This supports a CTR-fix style review action, such as improving the title, meta description, or search intent match.

In [8]:
# Baseline rule:
# Prioritize pages that are visible in search but have weak click performance.

baseline_df = work_df.copy()

# Higher when CTR is low
baseline_df["low_ctr_factor"] = 1 - baseline_df["gsc_ctr"].clip(0, 1)

# Higher when position is still reasonably visible
baseline_df["position_factor"] = np.where(
    baseline_df["gsc_avg_position"] <= 20,
    1,
    0.5
)

# Score combines visibility, weak CTR, and position
baseline_df["baseline_score"] = (
    np.log1p(baseline_df["gsc_impressions"])
    * baseline_df["low_ctr_factor"]
    * baseline_df["position_factor"]
)

baseline_df["reason_code"] = "visible_low_ctr"

baseline_df["action_label"] = "review_title_meta_and_search_intent"

ranked_queue = (
    baseline_df[
        [
            "content_hash_id",
            "report_date",
            "baseline_score",
            "reason_code",
            "action_label",
            "gsc_impressions",
            "gsc_clicks",
            "gsc_ctr",
            "gsc_avg_position",
            "ga4_sessions",
            "ga4_engaged_sessions",
        ]
    ]
    .sort_values("baseline_score", ascending=False)
    .reset_index(drop=True)
)

ranked_queue.head(10)

,content_hash_id,report_date,baseline_score,reason_code,action_label,gsc_impressions,gsc_clicks,gsc_ctr,gsc_avg_position,ga4_sessions,ga4_engaged_sessions
0,content_44f34c0a90047651,2026-03-28,10.598493,visible_low_ctr,review_title_meta_and_search_intent,40084,1,0.000025,0.083350,0.0,0.0
1,content_34a70fea29d15f24,2026-03-04,10.570877,visible_low_ctr,review_title_meta_and_search_intent,39003,2,0.000051,2.764916,NaN,NaN
2,content_945d6ff91386c817,2026-03-04,10.528597,visible_low_ctr,review_title_meta_and_search_intent,37368,0,0.000000,8.613948,NaN,NaN
3,content_eadb33b5df496f4a,2026-03-29,10.511305,visible_low_ctr,review_title_meta_and_search_intent,39305,252,0.006411,2.197507,122.0,15.0
4,content_eadb33b5df496f4a,2026-03-28,10.482343,visible_low_ctr,review_title_meta_and_search_intent,38436,271,0.007051,2.195988,141.0,7.0
5,content_fec55986a1868d62,2026-03-30,10.415832,visible_low_ctr,review_title_meta_and_search_intent,33383,0,0.000000,0.181500,0.0,0.0
6,content_eadb33b5df496f4a,2026-03-30,10.408040,visible_low_ctr,review_title_meta_and_search_intent,35404,225,0.006355,2.188397,104.0,3.0
7,content_44f34c0a90047651,2026-03-27,10.403020,visible_low_ctr,review_title_meta_and_search_intent,32958,0,0.000000,0.132532,2.0,0.0
8,content_44f34c0a90047651,2026-03-29,10.396237,visible_low_ctr,review_title_meta_and_search_intent,32756,2,0.000061,0.142508,3.0,0.0
9,content_eadb33b5df496f4a,2026-03-27,10.390908,visible_low_ctr,review_title_meta_and_search_intent,34817,223,0.006405,2.181348,116.0,4.0


In [9]:
output_dir = "work/outputs"
os.makedirs(output_dir, exist_ok=True)

csv_path = os.path.join(output_dir, "baseline_action_score.csv")

ranked_queue.to_csv(csv_path, index=False)

print("Wrote:", csv_path)
print("Rows written:", len(ranked_queue))

Wrote: work/outputs/baseline_action_score.csv
Rows written: 3611061


In [10]:
metrics = {
    "lane": "Lane 2 - Refresh / Content Opportunity Scoring",
    "month": "2026-03",
    "table": "fact_content_daily_performance",
    "rows_loaded": int(len(df)),
    "rows_after_gsc_available_and_impressions": int(len(work_df)),
    "baseline_rows_written": int(len(ranked_queue)),
    "signal_1": "volume",
    "signal_1_verdict": "CONFIRMED",
    "signal_2": "ctr_vs_position",
    "signal_2_verdict": "CONFIRMED",
    "rule_reason_code": "visible_low_ctr",
    "rule_action_label": "review_title_meta_and_search_intent",
}

json_path = os.path.join(output_dir, "w04_baseline_metrics.json")

with open(json_path, "w") as f:
    json.dump(metrics, f, indent=2)

print("Wrote:", json_path)
metrics

Wrote: work/outputs/w04_baseline_metrics.json


{'lane': 'Lane 2 - Refresh / Content Opportunity Scoring',
 'month': '2026-03',
 'table': 'fact_content_daily_performance',
 'rows_loaded': 9841378,
 'rows_after_gsc_available_and_impressions': 3611061,
 'baseline_rows_written': 3611061,
 'signal_1': 'volume',
 'signal_1_verdict': 'CONFIRMED',
 'signal_2': 'ctr_vs_position',
 'signal_2_verdict': 'CONFIRMED',
 'rule_reason_code': 'visible_low_ctr',
 'rule_action_label': 'review_title_meta_and_search_intent'}

In [11]:
top10 = ranked_queue.head(10).copy()

top10[
    [
        "content_hash_id",
        "report_date",
        "action_label",
        "reason_code",
        "baseline_score",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_ctr",
        "gsc_avg_position",
    ]
]

,content_hash_id,report_date,action_label,reason_code,baseline_score,gsc_impressions,gsc_clicks,gsc_ctr,gsc_avg_position
0,content_44f34c0a90047651,2026-03-28,review_title_meta_and_search_intent,visible_low_ctr,10.598493,40084,1,0.000025,0.083350
1,content_34a70fea29d15f24,2026-03-04,review_title_meta_and_search_intent,visible_low_ctr,10.570877,39003,2,0.000051,2.764916
2,content_945d6ff91386c817,2026-03-04,review_title_meta_and_search_intent,visible_low_ctr,10.528597,37368,0,0.000000,8.613948
3,content_eadb33b5df496f4a,2026-03-29,review_title_meta_and_search_intent,visible_low_ctr,10.511305,39305,252,0.006411,2.197507
4,content_eadb33b5df496f4a,2026-03-28,review_title_meta_and_search_intent,visible_low_ctr,10.482343,38436,271,0.007051,2.195988
5,content_fec55986a1868d62,2026-03-30,review_title_meta_and_search_intent,visible_low_ctr,10.415832,33383,0,0.000000,0.181500
6,content_eadb33b5df496f4a,2026-03-30,review_title_meta_and_search_intent,visible_low_ctr,10.408040,35404,225,0.006355,2.188397
7,content_44f34c0a90047651,2026-03-27,review_title_meta_and_search_intent,visible_low_ctr,10.403020,32958,0,0.000000,0.132532
8,content_44f34c0a90047651,2026-03-29,review_title_meta_and_search_intent,visible_low_ctr,10.396237,32756,2,0.000061,0.142508
9,content_eadb33b5df496f4a,2026-03-27,review_title_meta_and_search_intent,visible_low_ctr,10.390908,34817,223,0.006405,2.181348


## Top-10 review

1. Action: review title, meta description, and search intent. Why it is here: high search impressions, low CTR, and visible average position. What would make it wrong: the page may already be intentionally informational and not meant to earn many clicks.

2. Action: review title, meta description, and search intent. Why it is here: the baseline score found a visible page with weak click performance. What would make it wrong: the query mix may be branded, navigational, or not aligned with the page purpose.

3. Action: review title, meta description, and search intent. Why it is here: it has enough search visibility to make review worthwhile. What would make it wrong: the low CTR may be normal for this page type or search result layout.

4. Action: review title, meta description, and search intent. Why it is here: the page appears in search but does not convert visibility into clicks well. What would make it wrong: impressions may come from irrelevant queries that the page should not target.

5. Action: review title, meta description, and search intent. Why it is here: the rule prioritizes visible pages where click performance looks weak. What would make it wrong: the page may have been recently changed, and the current window may not reflect the update yet.

6. Action: review title, meta description, and search intent. Why it is here: high visibility and low CTR suggest possible title/meta mismatch. What would make it wrong: low CTR may be caused by SERP features, ads, or zero-click search behavior.

7. Action: review title, meta description, and search intent. Why it is here: the page has search exposure but weak click capture. What would make it wrong: the average position may hide many different queries with different intent.

8. Action: review title, meta description, and search intent. Why it is here: the rule flags it as a visible low-CTR opportunity. What would make it wrong: the page may be ranking for broad terms where low CTR is expected.

9. Action: review title, meta description, and search intent. Why it is here: the score suggests the page is worth checking before lower-visibility pages. What would make it wrong: the page may serve awareness or support content where clicks are not the main goal.

10. Action: review title, meta description, and search intent. Why it is here: the page matches the baseline rule: visible search performance with weak click response. What would make it wrong: the page may not need refresh if the low CTR is caused by search behavior outside the site owner’s control./

## Weak picks and limitation

One limitation of this baseline is that it uses daily page-level performance from March 2026 only. It does not include full content metadata such as page topic, title, meta description, word count, last updated date, or the queries that caused the impressions.

Because of this, the baseline may rank pages that look weak numerically but are not actually good refresh candidates. For example, low CTR may be caused by search intent, SERP features, ads, or broad queries rather than a bad title or bad content.

This is why the top-10 review matters. The baseline creates a queue, but a human still needs to inspect the page and context before deciding what to change.

## Self-check

- [x] I confirmed my lane: Lane 2, Refresh / Content Opportunity Scoring.
- [x] I checked two signals before writing the rule.
- [x] I used volume/impressions as one signal.
- [x] I used CTR-vs-position as one FlyRank-style flag-linked signal.
- [x] I printed bucket tables with `n`.
- [x] I gave each signal a one-word verdict.
- [x] I encoded one baseline rule.
- [x] I created a baseline score.
- [x] I created one reason code: `visible_low_ctr`.
- [x] I created one action label: `review_title_meta_and_search_intent`.
- [x] I wrote `work/outputs/baseline_action_score.csv`.
- [x] I reviewed the top 10 rows with what would make each one wrong.
- [x] I avoided future-window and label-derived inputs.